# 🆕 XFarmaCare — Clasificación de Nuevos Clientes
## Asignar segmento a clientes nuevos con el modelo entrenado

**Propósito:** Cuando un nuevo cliente acumula suficiente historial (≥3 compras),
este script lo clasifica en uno de los segmentos existentes usando el modelo
de clustering guardado. Se ejecuta como job recurrente.
---

In [ ]:
# === Setup ===
import pandas as pd
import numpy as np
import joblib, os, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

MODEL_DIR = "models/"
OUTPUT_DIR = "outputs/"

# Cargar componentes del modelo
modelo_cluster = joblib.load(f"{MODEL_DIR}modelo_clustering_xfarmacare.pkl")
scaler_cluster = joblib.load(f"{MODEL_DIR}scaler_clustering.pkl")
cluster_features = joblib.load(f"{MODEL_DIR}cluster_features.pkl")

print(f"[{datetime.now()}] Modelo de clustering cargado.")
print(f"K = {modelo_cluster.n_clusters} clusters")
print(f"Features: {len(cluster_features)}")

In [ ]:
# === Cargar datos del cliente nuevo (simulado) ===
# En producción, esto vendría de la base de datos.
# Aquí tomamos algunos clientes del dataset como ejemplo.

df = pd.read_csv(f"{OUTPUT_DIR}dataset_analitico_clientes.csv")

# Simular "nuevos" clientes: los últimos 200 registrados
nuevos = df.tail(200).copy()

# Preparar features
X_nuevo = nuevos[cluster_features].fillna(0)
X_nuevo_scaled = scaler_cluster.transform(X_nuevo)

# Predecir cluster
nuevos['cluster'] = modelo_cluster.predict(X_nuevo_scaled)

# Cargar perfil de clusters para asignar nombre
perfil = pd.read_csv(f"{OUTPUT_DIR}perfil_clusters.csv", index_col=0)

# Mapeo de cluster a nombre (basado en el entrenamiento)
segmentos_existentes = pd.read_csv(f"{OUTPUT_DIR}output_segmentos_clientes.csv")
mapeo = segmentos_existentes.groupby('cluster')['segmento'].first().to_dict()
nuevos['segmento'] = nuevos['cluster'].map(mapeo).fillna('Estándar')

print(f"Nuevos clientes clasificados: {len(nuevos)}")
print(f"\nDistribución:")
print(nuevos['segmento'].value_counts())

In [ ]:
# === Actualizar archivo de segmentos ===
output_nuevos = nuevos[['customer_id', 'cluster', 'segmento']].copy()
output_nuevos['fecha_segmentacion'] = datetime.now().strftime('%Y-%m-%d')

# Leer el existente y agregar (evitar duplicados)
existente = pd.read_csv(f"{OUTPUT_DIR}output_segmentos_clientes.csv")
combinado = pd.concat([existente, output_nuevos]).drop_duplicates(
    subset='customer_id', keep='last')
combinado.to_csv(f"{OUTPUT_DIR}output_segmentos_clientes.csv", index=False)

print(f"Archivo de segmentos actualizado: {len(combinado):,} clientes totales")
print(f"[{datetime.now()}] Scoring de clustering completado.")

## ✅ Clasificación Completada
Nuevos clientes asignados a su segmento correspondiente.
El archivo `output_segmentos_clientes.csv` queda actualizado.